In [1]:
import pandas as pd

df = pd.read_pickle("../data/processed/movies_features.pkl")

threshold = df["vote_count"].quantile(0.75)
df["high_engagement"] = (df["vote_count"] >= threshold).astype(int)

print("Threshold (75th percentile):", threshold)
print(df["high_engagement"].value_counts())

Threshold (75th percentile): 8566.5
high_engagement
0    1289
1     430
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

numeric_features = ["runtime", "budget", "revenue", "popularity", "n_genres", "n_keywords", "overview_length"]
categorical_features = ["original_language", "runtime_category"]

X = df[numeric_features + categorical_features]
y = df["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((1375, 9), (344, 9))

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [4]:
from sklearn.linear_model import LogisticRegression

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

log_reg_pipeline.fit(X_train, y_train)
print("Trained")

Trained


In [5]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.95      0.89       258
           1       0.77      0.47      0.58        86

    accuracy                           0.83       344
   macro avg       0.81      0.71      0.74       344
weighted avg       0.82      0.83      0.82       344

ROC-AUC: 0.8373895799531278
Confusion matrix:
 [[246  12]
 [ 46  40]]
